
# Arm scan: positioner 회전 중심 (과 arm 길이)

**초기 위치(기하각 alpha 30°, beta 150° = 명령각 (0, 0))에서 beta 는 두고 alpha 만 한 바퀴 돌리며** 찍은 이미지들로,
positioner 마다 fiber 끝의 궤적에 원을 맞춘다 (`armcal.py`). 원의 중심 = alpha 회전 중심 (Measured_Geometry 의 X, Y),
반지름 R = |a₁ + a₂e^{iφ}|, φ = 150° 이므로 설계값 (5.2, 11.6) 이면 R ≈ 7.6 mm.
한 beta 에서만 돌리면 a₁, a₂ 를 따로 가를 수는 없고, 바깥 arm a₂ 를 측정표 값으로 두면 `arm1_from_radius` 로 a₁ 이 나온다
(beta = 0 과 180 두 번 돌리면 `fit_arms` 로 둘 다).

프레임마다 본 metrology 와 같은 방법으로 fiducial 을 찾아 변환을 새로 맞추므로 (첫 프레임 패턴 매칭, 그 뒤는 직전 프레임의 fiducial 자리) scan 중 카메라가 흘러가도 된다.
결과는 `Measured_Geometry` 형식으로 써서 패키지의 실측 회전 중심으로 쓸 수 있다.


## 촬영 (take_exposure.py)
positioner 를 모두 초기 위치 (명령각 (0, 0)) 에 두고, fiducial 과 fiber 를 둘 다 켠 채로 **alpha 명령각만** 바꾸며 단계마다
세 장(fiber 30 ms + fiducial 1 ms, 10 ms)을 한 세트로 찍는다. beta 는 0 (명령각) 그대로.
`take_exposure.py` 의 세트 촬영 (`--fiber`, `--fid`) 을 쓰면 이름이 `{stem}_fib_30ms_0.fits`, `{stem}_fid1_1ms_0.fits`, `{stem}_fid2_10ms_0.fits` 로 붙는다.
이름에 각도를 넣으면 alpha 순으로 정렬된다 (음수 각도는 m 을 붙이는 식으로: alpha_m030).
```bash
for a in 000 030 060 090 120 150 180 210 240 270 300 330; do
    # positioner alpha 명령각을 $a 도로 보낸 뒤 (beta 명령각 0 유지)
    python kspec_metrology/notebooks/take_exposure.py --fiber 30 --fid 1 10 --out /data/scan_a/alpha_$a
done
```
→ 2절에서 `SCAN = armcal.scan_sets('/data/scan_a/alpha_*')`. 12 단계가 아니어도 되지만 원을 맞추려면 positioner 마다 5 점 이상, 한 바퀴에 고르게 퍼져 있는 것이 좋다.
alpha 가 positioner 의 명령 범위를 넘으면 그 각도는 빼면 된다 (반 바퀴만 돌아도 원은 맞는다).


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (-1, -1)        # 명령각 = 부호 x 기하각 - 영점. 본 장비 positioner 는 기하각(반시계 alpha, elbow-down beta)의 반대로 돈다 (2026-10-27 확인)
ANGLE_ZERO    = (-30, -150)     # positioner 명령각 영점: 기하각 (30, 150) = 명령 (0, 0) (부호 -1 이라 영점은 -30, -150). 출력·각도 입력 모두 명령각
MEASURED_POSITIONERS = True     # positioner 회전 중심을 arm scan 실측값(Measured_Geometry.txt)으로 쓸지. 핀홀 실측과는 별개 스위치 (2026-10-27 부터 기본 True)
ALPHA_RANGE   = (-330, 30)      # positioner 가 받는 alpha 명령각 범위 [lo, hi). 절대 명령각(Trial 0, 누적각)은 이 범위로 접는다

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_ANGLE_ZERO']    = f'{ANGLE_ZERO[0]},{ANGLE_ZERO[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'                        # fiducial 핀홀 실측 위치 + arm 길이 측정표 (거의 항상 켬)
os.environ['KSPEC_USE_MEASURED_POSITIONERS'] = '1' if MEASURED_POSITIONERS else '0'
os.environ['KSPEC_ALPHA_RANGE']   = f'{ALPHA_RANGE[0]},{ALPHA_RANGE[1]}'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN, '| angle zero', cfg.ANGLE_ZERO)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'


In [ ]:

from kspec_metrology.analysis import armcal
SCAN = armcal.scan_sets('/data/scan_a/alpha_*')             # [(fiber 30 ms, [fid 1 ms, fid 10 ms]), ...] alpha 순
print(len(SCAN), 'frames;', 'first:', [os.path.basename(p) for p in [SCAN[0][0]] + SCAN[0][1]])
res_scan = armcal.arm_calibration(SCAN, magnification=MAGNIFICATION, model=MODEL,
                                  n_extra=N_EXTRA, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
T = res_scan.table
print(f"fitted {int(np.isfinite(T.R).sum())}/{len(T)}; R median {T.R.median():.3f} mm; circle rms median {T.rms.median()*1e3:.0f} um; "
      f"centre - design: median {np.hypot(T.dx, T.dy).median()*1e3:.0f} um")
T.round(4)


In [ ]:

armcal.plot_arm_calibration(res_scan);



## 반지름 점검과 arm 길이, 실측 기하 파일
beta 가 150° (기하각) 로 고정되어 있으므로 R 은 `expected_radius(a1, a2, 150)` 와 비슷해야 한다 (측정표 arm 기준).
크게 다른 positioner 는 beta 영점이 다르거나 fiber 가 다른 positioner 의 것으로 잘못 붙은 것이다 (3절 그림에서 확인).


In [ ]:

a1_tab, a2_tab = cfg.arm_lengths()                                   # 측정표 arm (Positioner_Map 기준)
T = res_scan.table.copy()
T['R_expect'] = armcal.expected_radius(a1_tab, a2_tab, 150.)        # 기하각 beta = 150 에서의 기대 반지름
T['a1_from_R'] = armcal.arm1_from_radius(T.R.values, a2_tab, 150.)  # a2 를 측정표 값으로 두고 R 에서 푼 a1
T['dR_um'] = (T.R - T.R_expect)*1e3
print(f"R - R_expect: median {T.dR_um.median():.0f} um, |max| {T.dR_um.abs().max():.0f} um")
display(T[['x0', 'y0', 'R', 'R_expect', 'dR_um', 'a1_from_R', 'rms', 'n', 'dx', 'dy']].round(4))

arms = None            # 회전 중심만 쓴다. a1_from_R 을 믿을 수 있으면: arms = {i: (a, b) for i, a, b in zip(T.index, T.a1_from_R, a2_tab)}
out = armcal.write_measured_geometry(res_scan, os.path.join(OUT_DIR, 'Measured_Geometry_scan.txt'),
                                     template=os.path.join(os.path.dirname(cfg.__file__), 'Measured_Geometry.txt'), arms=arms)
print(out); print(open(out).read()[:800])
